# Course 1, Week 3 — Data Management in PyTorch

- [DeepLearning.AI platform](https://learn.deeplearning.ai/specializations/pytorch-for-deep-learning-professional-certificate/lesson)
- [Week notes](README.md)
- [GitHub issue #3](https://github.com/majorgilles/pytorch_for_deep_learning/issues/3)

**Focus:** Represent datasets and feed repeatable batches through Dataset and DataLoader.


## Introduction to data pipelines

> [Open this lesson on the DeepLearning.AI learning platform](https://learn.deeplearning.ai/specializations/pytorch-for-deep-learning-professional-certificate/lesson/bvvjkv/introduction-to-data-pipelines)

The first two modules established the basic workflow: load data, define a model, train it, and produce predictions. Those steps worked with MNIST because its images and labels were already organized consistently. Real projects often fail earlier—not because the model is weak, but because the data pipeline delivers the wrong, inconsistent, or inefficiently loaded inputs.

### The botanical-garden challenge

A botanical garden wants a flower-identification application trained with the Oxford 102 Flowers dataset. Several teams can use the same architecture and still obtain very different results when they prepare the data differently. This makes the pipeline itself part of the model-building problem.

The raw dataset introduces practical complications:

- Images are stored as generically named JPEG files rather than class-specific folders.
- Class labels are stored separately in a MATLAB `.mat` file.
- Each filename must be matched with the correct label before training.
- Images must be converted to consistent sizes, data types, channel layouts, and tensor shapes.
- Samples must be loaded in batches without making storage access the training bottleneck.

For image models, preprocessing must ultimately produce batches with a consistent shape such as $[B,C,H,W]$. Here, $B$ is the batch size, $C$ is the channel count, and $H$ and $W$ are the transformed image dimensions. Images with incompatible heights or widths cannot be stacked into one ordinary batch without first being resized, cropped, or padded.

### Three categories of data-pipeline problems

| Category | Core question | Typical failure |
|---|---|---|
| **Access** | Can each input be found and paired with its label? | An image is missing, unreadable, or matched to the wrong class. |
| **Quality** | Is each sample valid and consistently prepared? | Shapes, data types, channels, or labels differ unexpectedly. |
| **Efficiency** | Can training receive batches fast enough? | Loading samples one at a time leaves the model waiting for data. |

A useful mental model for the complete flow is:

$$\text{image files + label metadata} \longrightarrow \text{sample mapping} \longrightarrow \text{transforms} \longrightarrow \text{Dataset} \longrightarrow \text{DataLoader} \longrightarrow \text{model}.$$

### What this module develops

This module revisits `Dataset` and `DataLoader` in greater depth. The goal is to learn how to:

1. Access image files and labels stored in separate formats.
2. Validate and transform samples into model-ready tensors.
3. Implement dataset indexing and length behavior.
4. Assemble shuffled, repeatable batches efficiently.
5. Diagnose whether failures come from access, quality, or efficiency.

A sophisticated network cannot compensate for mislabeled, malformed, or poorly served data. A reliable pipeline makes every later training result easier to trust.


## Solving data-access problems with a custom `Dataset`

The first pipeline challenge is reliable access. The Oxford 102 Flowers archive contains $8{,}189$ images in one flat directory, with generic numbered filenames. Its $8{,}189$ labels live separately in a MATLAB `.mat` file and represent 102 flower classes. Unlike a pre-built MNIST dataset, this raw layout does not tell PyTorch how an image and its label belong together.

A custom `Dataset` supplies that missing contract through three methods:

| Method | Responsibility | Oxford Flowers behavior |
|---|---|---|
| `__init__` | Record the information needed later. | Store the image directory, read label metadata, convert labels to zero-based class indices, and retain an optional transform. |
| `__len__` | Report the number of samples. | Return $8{,}189$, normally derived from the label count. |
| `__getitem__` | Load one indexed sample. | Resolve one filename, open that image, find its matching label, apply the transform, and return `(image, label)`. |

### Keep three index systems separate

This dataset contains two independent off-by-one hazards:

| Meaning | Stored range | Conversion from dataset index $i$ |
|---|---:|---|
| Python dataset index | $0 \ldots 8{,}188$ | $i$ |
| Image file number | $1 \ldots 8{,}189$ | $i+1$ |
| Raw MATLAB class label | $1 \ldots 102$ | `raw_labels[i]` |
| PyTorch class index | $0 \ldots 101$ | `raw_labels[i] - 1` |

The filename formatter `:05d` pads the one-based file number to five digits. Dataset index 0 must therefore resolve to `image_00001.jpg`, not the nonexistent `image_00000.jpg`. Label conversion is separate: subtracting one from a raw label changes the class index, not the filename.

### Lazy loading

`__init__` should load lightweight metadata, not all image pixels. The actual image is opened only when `__getitem__(i)` is called:

$$i \longrightarrow \text{filename}_{i+1} \longrightarrow \text{open image} \longrightarrow \text{transform} \longrightarrow (x_i,y_i).$$

Before batching, one transformed sample commonly has image shape $[C,H,W]$ and a scalar class label. `DataLoader` later stacks compatible samples into image batches shaped $[B,C,H,W]$ and target batches shaped $[B]$. Lazy loading keeps memory use tied mainly to the active batch rather than the full archive.

### Test access before training

Check the dataset contract as soon as it is implemented:

1. Confirm the number of image paths equals the number of labels.
2. Load the first and last samples to expose indexing mistakes.
3. Verify class indices satisfy $0 \le y < 102$.
4. Inspect image mode, data type, and shape after transformation.
5. Display a few image-label pairs to catch systematic mismatches.

Once access is trustworthy, the next pipeline concern is quality: converting images with different sizes, modes, and other properties into consistent model-ready tensors.


In [1]:
from pathlib import Path

import torch


def flower_image_path(image_directory: Path, index: int) -> Path:
    """Map a zero-based dataset index to its one-based Oxford image filename."""
    return image_directory / f"image_{index + 1:05d}.jpg"


image_directory: Path = Path("data/flowers/jpg")
raw_labels: torch.Tensor = torch.tensor([1, 1, 102], dtype=torch.int64)
class_indices: torch.Tensor = raw_labels - 1

print(f"dataset index 0 -> {flower_image_path(image_directory, 0).name}")
print(f"raw labels: {raw_labels.tolist()}")
print(f"class indices: {class_indices.tolist()}")

assert flower_image_path(image_directory, 0).name == "image_00001.jpg"
assert class_indices.tolist() == [0, 0, 101]

dataset index 0 -> image_00001.jpg
raw labels: [1, 1, 102]
class indices: [0, 0, 101]
